# Evaluating LLM Outputs using LLUMO API


This notebook evaluates the quality of multiple LLM-generated responses based on key metrics such as confidence, relevance, and accuracy etc. The evaluation is performed using the LLUMO API.

- OpenAI: This is the official OpenAI Python client, used to interact with - - OpenAI's API.
- requests: A popular library for making HTTP requests in Python.
- json: Used for parsing JSON data, which is common in API responses.
- from google.colab : Used to securely store and retrieve user-specific data in Google Colab.

In [5]:
%pip install openai
import os
import openai
import requests
from google.colab import userdata

# Secure API key handling

In [23]:
# Fetch API keys
OPENAI_KEY = userdata.get("OPEN_API_KEY")
LLUMOAI_KEY = userdata.get("LLUMO_API_KEY")

if not OPENAI_KEY:
    raise ValueError("Missing OpenAI API key. Set OPENAI_API_KEY as an environment variable.")

# Initialize OpenAI client
client = openai.Client(api_key=OPENAI_KEY)

# Define OpenAi function

In [24]:

# Function to generate response from OpenAI
def get_response(prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {str(e)}"


# Getting Respone from OpenAI &  Evaluate with LLumo AI

In [40]:
# Function to evaluate responses with LLumo AI
def evaluate_response(prompt, openai_response):
    LLUMO_ENDPOINT = "https://app.llumo.ai/api/create-eval-analytics"
    headers = {
        "Authorization": f"Bearer {LLUMOAI_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "prompt": "Give any for the query: {query}, in a way that you are explaining to a 10 year old kid.",
        "input": {"query": prompt},
        "output": f'{openai_response}',
        "analytics": ["Confidence"]
    }
    try:
        response = requests.post(LLUMO_ENDPOINT, json=payload, headers=headers)
        result = response.json()
        print("Response from LLumo:", result)
        if 'data' in result:
            print("statusCode :", result['data'].get('statusCode', 'N/A'))
            print("message :", result['data'].get('message', 'N/A'))
            return result.get('data', {})
        else:
            return {"error": "Unexpected response format", "details": result}
    except Exception as e:
        return {"error": str(e)}

# List of prompts
prompts = [
    "Explain quantum physics in a way that's easy to understand.",
    "Describe the significance of the Turing test in AI.",
    "Summarize the theory of relativity in simple terms.",
    "How does blockchain technology work?",
    "What are the key differences between supervised and unsupervised learning?"
]

# Iterate through each prompt
for prompt in prompts:
    print(f"Processing prompt: {prompt}\n")
    openai_response = get_response(prompt)
    print(f"OpenAI Response: {openai_response}\n")

    evaluation = evaluate_response(prompt, openai_response)
    print(f"LLumo Evaluation: {evaluation}\n")
    print("---------------------------------------------\n")


Processing prompt: Explain quantum physics in a way that's easy to understand.

OpenAI Response: Quantum physics, also known as quantum mechanics, is a branch of physics that deals with phenomena on a very small scale, such as molecules, atoms, and atomic particles like electrons, protons, and photons (particles of light).

Here are a few key concepts explained in simple terms:

1. **Wave-Particle Duality**: Everything in the universe has both particle and wave properties. For example, light can behave like particles (which we call photons) and also like waves (ripples or oscillations).

2. **Superposition**: Particles can be in multiple states at once. Imagine if you were both sitting and standing at the same time - that's superposition!

3. **Quantum Entanglement**: This is a special connection between particles, where the state of one particle instantly affects the state of another, no matter how far apart they are. It's like twin telepathy in the microscopic world.

4. **Uncertaint